# 비전 모델 경량화 — torchvision 양자화 & TensorRT (Jetson Orin Nano)

이 노트북은 LLM 랩(Day 1~6)과 **별개 트랙**으로, 비전 모델(ResNet-18)을 두 가지 방식으로 경량화하고 이 Jetson에서 **실제 속도/크기**를 측정합니다.

- **트랙 A — PyTorch 공식 방식의 INT8 양자화 (CPU)**: 개념 학습용. torchvision 양자화 모델 + eager-mode post-training quantization(PTQ).
- **트랙 B — TensorRT (PyTorch → ONNX → trtexec)**: Jetson **GPU 가속**의 정석. FP32/FP16/INT8 엔진을 빌드해 속도 비교.

> 참고 공식 자료
> - PyTorch: [Quantized Transfer Learning for CV](https://docs.pytorch.org/tutorials/intermediate/quantized_transfer_learning_tutorial.html), [Torchvision Quantized Models](https://docs.pytorch.org/vision/main/models/resnet_quant.html)
> - 이 노트북의 모든 코드는 Jetson Orin Nano(JetPack 7.2, torch 2.13.0+cu130, TensorRT 10.16)에서 실제 실행·검증되었습니다.

### 이 Jetson에서 배운 핵심 결론 (미리 보기)
- CPU INT8 양자화는 모델 **크기는 ~1/4로 줄지만**, 이 기기 CPU에서는 오히려 **느립니다**. → 양자화가 곧 속도 향상은 아니다(백엔드/하드웨어에 달림).
- 진짜 속도는 **GPU + TensorRT**에서 나옵니다: INT8 엔진이 PyTorch eager 대비 **~10배** 빠릅니다.

## 0. 환경 준비

이 랩은 torch/torchvision 외에 ONNX 내보내기 도구가 필요합니다. 이 Jetson 환경 특성상 주의점:
- **torchvision은 `--no-deps`로 설치**해야 합니다. 그냥 `pip install torchvision` 하면 의존성 해결 과정에서 특수 빌드된 `torch 2.13.0+cu130`(Jetson CUDA 빌드)를 일반 빌드로 덮어써 CUDA가 깨질 수 있습니다.
- torch 2.13의 ONNX export는 `onnxscript`를 요구합니다.
- TensorRT `trtexec`는 JetPack에 기본 포함되어 `/usr/bin/trtexec`에 있습니다(별도 설치 불필요).

In [11]:
import subprocess, sys, shutil, importlib.util, os

def ensure(pkg, pip_name=None, no_deps=False):
    if importlib.util.find_spec(pkg) is None:
        args = [sys.executable, "-m", "pip", "install", "-q", pip_name or pkg]
        if no_deps:
            args.insert(4, "--no-deps")
        print("installing", pip_name or pkg, "...")
        subprocess.run(args, check=True)
    else:
        print(pkg, "OK")

ensure("torchvision", no_deps=True)   # torch를 건드리지 않도록 --no-deps
ensure("onnx")
ensure("onnxscript")

print("trtexec:", shutil.which("trtexec") or "/usr/bin/trtexec")
assert os.path.exists("/usr/bin/trtexec"), "trtexec 없음 (JetPack TensorRT 확인 필요)"

WORK = os.path.expanduser("~/vision_lab")   # 커널이 Jetson에서 도므로 Jetson 절대경로 사용
os.makedirs(WORK, exist_ok=True)
print("작업 디렉토리:", WORK)

torchvision OK
onnx OK
onnxscript OK
trtexec: /usr/bin/trtexec
작업 디렉토리: /home/manager/vision_lab


In [12]:
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cuda", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
# CC 8.7 미지원 경고가 떠도 정상 (Day1 참고: PTX JIT로 대체 실행됨)

torch 2.13.0+cu130 | torchvision 0.28.0+cu130 | cuda True
GPU: Orin


## 트랙 A — INT8 양자화 (PyTorch 공식 PTQ, CPU)

### A-1. 이 Jetson에서의 함정: 양자화 백엔드
PyTorch 양자화 연산은 백엔드가 필요합니다: x86은 `fbgemm`/`x86`, ARM은 `qnnpack`. 이 torch 빌드의 기본 엔진은 `x86`이라, aarch64인 Jetson에서 그대로 쓰면 `RuntimeError: unknown architecture`가 납니다. **`qnnpack`으로 명시**해야 동작합니다.

In [13]:
import torch
print("기본 엔진:", torch.backends.quantized.engine)
print("지원 엔진:", torch.backends.quantized.supported_engines)
torch.backends.quantized.engine = "qnnpack"   # ARM 백엔드 (이걸 안 하면 Jetson에서 실패)
print("-> 설정:", torch.backends.quantized.engine)

기본 엔진: qnnpack
지원 엔진: ['qnnpack', 'onednn', 'x86', 'fbgemm']
-> 설정: qnnpack


### A-2. 공식 PTQ 흐름: fuse → qconfig → prepare → calibrate → convert

torchvision의 *quantizable* ResNet 아키텍처를 쓰되 표준 사전학습 float 가중치를 얹고, post-training static quantization을 직접 수행합니다. (이 흐름이 PyTorch 공식 양자화 튜토리얼의 핵심입니다.)

In [14]:
import torch, time
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.quantization import resnet18 as qresnet18

def sd_size_mb(sd):
    return sum(v.numel()*v.element_size() for v in sd.values() if hasattr(v, "numel")) / 1024**2

# quantizable 아키텍처 + 표준 float 사전학습 가중치
qm = qresnet18(weights=None, quantize=False).eval()
float_sd = ResNet18_Weights.DEFAULT.get_state_dict(progress=False)
missing, unexpected = qm.load_state_dict(float_sd, strict=False)
print("float 가중치 로드: missing", len(missing), "unexpected", len(unexpected))

# PTQ 5단계
qm.fuse_model()                                                  # 1) Conv-BN-ReLU 융합
qm.qconfig = torch.ao.quantization.get_default_qconfig("qnnpack")# 2) qconfig (ARM)
torch.ao.quantization.prepare(qm, inplace=True)                 # 3) observer 삽입
with torch.no_grad():                                            # 4) 캘리브레이션
    for _ in range(5):
        qm(torch.randn(8, 3, 224, 224))
torch.ao.quantization.convert(qm, inplace=True)                # 5) INT8 변환
print("INT8 변환 완료, 크기:", round(sd_size_mb(qm.state_dict()), 1), "MB")

float 가중치 로드: missing 0 unexpected 0


/tmp/ipykernel_521711/701951235.py:17: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.ao.quantization.prepare(qm, inplace=True)                 # 3) observer 삽입
/tmp/ipykernel_521711/701951235.py:21: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quanti

INT8 변환 완료, 크기: 10.7 MB


In [15]:
# FP32(CPU) vs INT8(CPU) 속도/크기 비교
fp32_cpu = resnet18(weights=ResNet18_Weights.DEFAULT).eval()
x = torch.randn(16, 3, 224, 224)

def bench_cpu(model, x, warmup=2, iters=10):
    with torch.no_grad():
        for _ in range(warmup): model(x)
        t0 = time.perf_counter()
        for _ in range(iters): model(x)
    return (time.perf_counter() - t0) / iters * 1000

fp32_ms = bench_cpu(fp32_cpu, x)
int8_ms = bench_cpu(qm, x)
print(f"FP32 CPU: {fp32_ms:6.1f} ms/batch(16)   size {sd_size_mb(fp32_cpu.state_dict()):.1f} MB")
print(f"INT8 CPU: {int8_ms:6.1f} ms/batch(16)   size {sd_size_mb(qm.state_dict()):.1f} MB")
print(f"크기 축소: {sd_size_mb(fp32_cpu.state_dict())/sd_size_mb(qm.state_dict()):.1f}x, "
      f"속도 배율: {fp32_ms/int8_ms:.2f}x")

FP32 CPU:  778.1 ms/batch(16)   size 44.6 MB
INT8 CPU:  906.7 ms/batch(16)   size 10.7 MB
크기 축소: 4.2x, 속도 배율: 0.86x


### A-3. 관찰 — "양자화 = 무조건 빠름"이 아니다

실측 예시 (이 Jetson, 실행마다 다소 변동):

```
FP32 CPU:  734.1 ms/batch(16)   size 44.6 MB
INT8 CPU:  905.7 ms/batch(16)   size 10.7 MB
크기 축소: 4.2x, 속도 배율: 0.81x  (INT8이 오히려 느림!)
```

- **크기는 ~4배 줄었습니다** (44.6 → 10.7 MB). 저장/전송/메모리 관점의 이득은 확실합니다.
- 그런데 **속도는 INT8이 더 느립니다.** Jetson의 Cortex-A78AE CPU에서 qnnpack INT8 커널이 이 모델/배치 조합에서 FP32 대비 이득을 못 냈기 때문입니다. INT8 이득은 하드웨어에 최적화된 커널이 있을 때만 나옵니다.
- 게다가 **torchvision 양자화 모델은 CPU 전용**입니다(GPU 미지원). Jetson의 진짜 무기인 GPU를 전혀 못 씁니다.

→ 결론: 엣지에서 "경량화로 빠르게"를 원한다면 CPU INT8이 아니라 **GPU + TensorRT**로 가야 합니다. 트랙 B에서 확인합니다.

## 트랙 B — TensorRT (PyTorch → ONNX → trtexec, GPU)

TensorRT는 NVIDIA GPU 전용 추론 최적화 엔진으로, JetPack에 기본 포함됩니다. 흐름은:
`PyTorch 모델 → ONNX 내보내기 → trtexec로 TensorRT 엔진 빌드(+정밀도 선택) → 벤치마크`.

### B-1. 먼저 PyTorch FP32 GPU eager 속도 (비교 기준)

In [16]:
import torch, time
from torchvision.models import resnet18, ResNet18_Weights

m_gpu = resnet18(weights=ResNet18_Weights.DEFAULT).eval().to("cuda")
xg = torch.randn(16, 3, 224, 224, device="cuda")
with torch.no_grad():
    for _ in range(3): m_gpu(xg)      # 워밍업 (CC8.7 PTX JIT 비용 제거 — Day1/Day2 참고)
torch.cuda.synchronize()
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(20): m_gpu(xg)
torch.cuda.synchronize()
eager_ms = (time.perf_counter() - t0) / 20 * 1000
print(f"PyTorch FP32 eager (GPU): {eager_ms:.2f} ms/batch(16)")

PyTorch FP32 eager (GPU): 56.05 ms/batch(16)


### B-2. ONNX로 내보내기

torch 2.13의 exporter는 큰 가중치를 **외부 데이터 파일**(`*.onnx.data`)로 분리 저장합니다. 그래서 `.onnx`와 `.onnx.data` **두 파일이 항상 같은 폴더에** 있어야 trtexec가 읽습니다.

> ⚠️ **opset 버전 함정**: `opset_version=17`을 지정해도 이 torch 2.13 exporter는 실제로 **opset 18로 내보냅니다**(`Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for.`). 18→17 다운그레이드도 시도하지만, adaptive average pooling이 만드는 `ReduceMean`의 `axes`가 opset 18에서 속성(attribute) 대신 입력(input)으로 바뀐 스펙 차이 때문에 실패합니다(`axes_input_to_attribute` 어설션 에러 — 콘솔에 `Traceback`이 찍히지만 **내부에서 잡힌 예외**라 export 자체는 계속 진행됩니다). 결과적으로 파일은 **opset 18로 저장**됩니다.
>
> **동작에는 문제 없습니다** — TensorRT 10.16은 opset 18을 지원하고 실제로 엔진 3개가 정상 빌드됩니다. 다만 코드의 `opset_version=17`은 실제 산출물과 다르므로, 아래 B-2.5 셀에서 `opset:` 출력이 18로 나오는 게 정상입니다. 헷갈리지 않으려면 `opset_version=18`로 맞춰 쓰는 편이 낫습니다.

In [17]:
import torch, os
from torchvision.models import resnet18, ResNet18_Weights

onnx_path = os.path.join(WORK, "resnet18_bs16.onnx")
m_cpu = resnet18(weights=ResNet18_Weights.DEFAULT).eval()
dummy = torch.randn(16, 3, 224, 224)
torch.onnx.export(m_cpu, dummy, onnx_path,
                  input_names=["input"], output_names=["output"], opset_version=17)
print("ONNX:", onnx_path)
for f in os.listdir(WORK):
    if f.startswith("resnet18_bs16.onnx"):
        print(" ", f, round(os.path.getsize(os.path.join(WORK, f))/1024**2, 1), "MB")

W0825 10:50:36.803000 521711 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/manager/mlops-lab-env/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/manager/mlops-lab-env/lib/python3.12/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/home/manager/mlops-lab-env/lib/python3.12/site-packages/onnxsc

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX: /home/manager/vision_lab/resnet18_bs16.onnx
  resnet18_bs16.onnx 0.1 MB
  resnet18_bs16.onnx.data 44.6 MB


### B-2.5. 내보낸 ONNX 그래프 들여다보기

`.onnx`는 그냥 바이너리가 아니라 **연산 그래프**입니다. 열어서 확인할 가치가 있는 것 세 가지:

1. **입력 shape이 정적인가** — 셀 B-2에서 `dynamic_axes`를 주지 않았으므로 배치 `16`이 숫자로 박혀 있어야 합니다. (동적이면 `'batch'` 같은 문자열로 나옵니다.) 이 엔진이 16장 배치 전용인 근거가 여기입니다.
2. **BatchNormalization이 남아 있는가** — 목록에 없다면 exporter가 이미 Conv에 접어 넣은 것입니다. 트랙 A에서 `fuse_model()`로 손수 했던 Conv-BN 융합을 ONNX export는 **자동으로** 해줍니다.
3. **노드가 몇 개인가** — 이 개수가 대략 eager 모드에서 띄우는 커널 수입니다. TensorRT는 이걸 훨씬 적은 수로 융합하며, 그게 FP32만으로도 2배가 나오는 이유입니다.

In [18]:
# ONNX 그래프 요약 — 무엇이 어떤 연산으로 번역됐나
import onnx, collections

model = onnx.load(onnx_path)            # .onnx.data(외부 가중치)도 자동으로 함께 읽음
onnx.checker.check_model(onnx_path)     # 규격 위반 없는지 검증 (통과하면 조용히 넘어감)
print("opset:", model.opset_import[0].version, "| producer:", model.producer_name)

def shape_of(t):                        # dim_value=고정 크기, dim_param=동적 축 이름
    return [d.dim_value or d.dim_param for d in t.type.tensor_type.shape.dim]

print("입력:", [(i.name, shape_of(i)) for i in model.graph.input])
print("출력:", [(o.name, shape_of(o)) for o in model.graph.output])

ops = collections.Counter(n.op_type for n in model.graph.node)
print()
print(f"총 노드 {len(model.graph.node)}개, 연산 종류 {len(ops)}가지")
for op, c in ops.most_common():
    print(f"  {op:22s} {c}")
print()
print("BatchNormalization 남아있나:", "Yes" if "BatchNormalization" in ops else "No (Conv에 융합됨)")

opset: 18 | producer: pytorch
입력: [('input', [16, 3, 224, 224])]
출력: [('output', [16, 1000])]

총 노드 49개, 연산 종류 7가지
  Conv                   20
  Relu                   17
  Add                    8
  MaxPool                1
  ReduceMean             1
  Reshape                1
  Gemm                   1

BatchNormalization 남아있나: No (Conv에 융합됨)


#### (선택) Netron으로 그래프를 그림으로 보기

텍스트 요약으로 충분하지만, residual connection이 갈라지고 합쳐지는 모양을 눈으로 보고 싶다면
[Netron](https://netron.app)을 Jetson에서 서버로 띄워 Windows 브라우저에서 접속할 수 있습니다.
노드를 클릭하면 커널 크기·stride·가중치 shape까지 나옵니다.

> ⚠️ 아래 셀은 **서버를 계속 띄워둡니다**(백그라운드 스레드). 다 봤으면 `netron.stop()`으로 내려주세요.
> 노트북에는 링크만 남고 그림은 저장되지 않으므로, 기록용 확인은 위의 텍스트 요약 셀이 담당합니다.

In [19]:
# (선택) Netron 서버 — 다 보고 나면 netron.stop() 실행
import subprocess
ensure("netron")                                 # 셀 0의 헬퍼 (torch를 건드리지 않는 순수 패키지)
import netron

host, port = netron.start(onnx_path, address=("0.0.0.0", 8080), browse=False)
ip = subprocess.run(["hostname", "-I"], capture_output=True, text=True).stdout.split()[0]
print(f"Windows 브라우저에서 접속:  http://{ip}:{port}")
print("종료: netron.stop()")

netron OK


OSError: [Errno 98] Address already in use

### B-3. TensorRT 엔진 빌드 + 벤치마크 (FP32 / FP16 / INT8)

`trtexec`가 엔진 빌드와 벤치마크를 함께 수행합니다. 각 엔진의 **GPU Compute Time(mean)**을 파싱해 비교합니다.

> ⚠️ 여기서 INT8은 `--int8`만 주어 trtexec가 자동 생성한 dynamic range를 씁니다 — **속도 측정용**입니다. 실제 정확도를 지키려면 대표 데이터로 **캘리브레이션 캐시**를 만들어야 합니다(아래 '직접 해보기' 참고).

In [20]:
import subprocess, os, re

def build_and_bench(name, extra_flags):
    engine = os.path.join(WORK, f"resnet18_{name}.engine")
    log = os.path.join(WORK, f"trt_{name}.log")
    cmd = ["/usr/bin/trtexec", f"--onnx={onnx_path}", f"--saveEngine={engine}"] + extra_flags
    with open(log, "w") as f:
        subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    text = open(log).read()
    m = re.search(r"GPU Compute Time:.*?mean = ([\d.]+) ms", text)
    ms = float(m.group(1)) if m else None
    size = os.path.getsize(engine)/1024**2 if os.path.exists(engine) else None
    passed = "PASSED" in text
    print(f"TRT {name:5s}: {ms} ms/batch(16)   engine {size:.1f} MB   {'OK' if passed else 'FAIL'}")
    return {"name": f"TRT {name}", "ms": ms, "size_mb": size}

trt_results = []
trt_results.append(build_and_bench("fp32", []))
trt_results.append(build_and_bench("fp16", ["--fp16"]))
trt_results.append(build_and_bench("int8", ["--int8"]))

TRT fp32 : 28.1899 ms/batch(16)   engine 44.8 MB   OK
TRT fp16 : 10.6277 ms/batch(16)   engine 22.5 MB   OK
TRT int8 : 5.396 ms/batch(16)   engine 11.4 MB   OK


### B-3.5. 엔진 안을 들여다보기 — 레이어 융합과 시간 배분

앞에서 "TensorRT가 여러 노드를 하나로 융합해서 빨라진다"고 했는데, 이걸 **숫자로 확인**합니다.
`--dumpProfile`은 이미 빌드된 엔진을 `--loadEngine`으로 불러 **레이어별 소요 시간**을 찍어줍니다.
빌드를 다시 하지 않으므로 엔진당 수 초면 끝납니다.

읽을 포인트:

1. **레이어 개수** — 셀 B-2.5의 ONNX 노드 수보다 훨씬 적어야 합니다. 그 차이가 곧 융합의 양이고,
   커널 실행 횟수와 중간 텐서 메모리 왕복이 그만큼 줄었다는 뜻입니다.
2. **시간을 잡아먹는 레이어** — 상위 5개가 전체의 몇 %인지. 최적화는 여기부터 손대야 합니다.
3. **`Reformatting`/`Reformat` 레이어** — 정밀도나 메모리 레이아웃을 바꾸느라 생긴 **순수 오버헤드**입니다.
   INT8 엔진에서 이게 늘어나기 쉬운데, 그만큼 INT8의 이론적 이득을 깎아먹습니다.

In [24]:
# 엔진 내부 레이어별 프로파일 — TensorRT가 ONNX 노드를 얼마나 융합했나 (재빌드 없음)
import subprocess, os, re

def run_profile(name, iters=100):
    engine = os.path.join(WORK, f"resnet18_{name}.engine")
    log = os.path.join(WORK, f"prof_{name}.log")
    cmd = ["/usr/bin/trtexec", f"--loadEngine={engine}",
           "--dumpProfile", "--dumpLayerInfo",
           "--profilingVerbosity=detailed", f"--iterations={iters}"]
    with open(log, "w") as f:                      # 빌드 없이 로드만 하므로 수 초면 끝남
        subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    return log

# trtexec 프로파일 표 형식 (TensorRT 10.x):
#   Time(ms)   Avg.(ms)   Median(ms)   Time(%)   Layer
#   452.73      3.9368      3.9013      14.2     node_Conv_291 + node_relu
# 숫자 4컬럼이 앞, 레이어 이름이 뒤(공백 포함).
ROW = re.compile(r"^([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+(\S.*)$")

def parse_profile(log):
    text = open(log, errors="replace").read()
    i = text.find("=== Profile")
    rows = []
    for line in (text[i:] if i >= 0 else text).splitlines():
        s = re.sub(r"^\s*\[[^\]]*\]\s*", "", line)     # [08/25/2026-11:20:00] 제거
        s = re.sub(r"^\s*\[[A-Z]\]\s*", "", s).strip() # [I] / [W] 제거
        m = ROW.match(s)
        if m and m.group(5).strip().lower() != "total":    # 마지막 Total 행은 제외
            rows.append((m.group(5).strip(), float(m.group(3)), float(m.group(4))))
    return rows                                        # (이름, median ms, 시간 비중 %)

prof = {}
for name in ("fp32", "fp16", "int8"):
    rows = parse_profile(run_profile(name))
    prof[name] = rows
    total = sum(r[1] for r in rows)
    reform = [r for r in rows if "reformat" in r[0].lower()]
    print(f"TRT {name}: 융합 후 레이어 {len(rows)}개, median 합계 {total:.2f} ms")
    print(f"   Reformatting {len(reform)}개 = {sum(r[2] for r in reform):.1f}% (순수 오버헤드)")
    for lname, med, pct in sorted(rows, key=lambda r: -r[2])[:5]:
        print(f"   {pct:5.1f}%  {med:7.3f} ms   {lname[:62]}")
    print()

try:
    n, k = len(model.graph.node), len(prof["int8"])
    print(f"ONNX 노드 {n}개 -> TRT INT8 레이어 {k}개  (융합률 {n / k:.1f}:1)")
    fused = [r[0] for r in prof["int8"]
             if "+" in r[0] and "reformat" not in r[0].lower()]
    print(f"'+'로 융합된 레이어 {len(fused)}개, 예: {fused[0][:70] if fused else '-'}")
except NameError:
    print("셀 B-2.5(ONNX 요약)를 먼저 실행하면 융합률까지 비교됩니다")

TRT fp32: 융합 후 레이어 24개, median 합계 27.54 ms
   Reformatting 1개 = 2.8% (순수 오버헤드)
    14.2%    3.901 ms   node_Conv_291 + node_relu
     8.0%    2.207 ms   node_Conv_295 + node_add + node_relu_2
     7.9%    2.190 ms   node_Conv_299 + node_add_1 + node_relu_4
     7.6%    2.081 ms   node_Conv_293 + node_relu_1
     7.5%    2.078 ms   node_Conv_297 + node_relu_3

TRT fp16: 융합 후 레이어 25개, median 합계 10.34 ms
   Reformatting 1개 = 3.2% (순수 오버헤드)
    11.0%    1.152 ms   node_Conv_291 + node_relu
     6.6%    0.688 ms   node_Conv_295 + node_add + node_relu_2
     6.6%    0.690 ms   node_Conv_299 + node_add_1 + node_relu_4
     5.9%    0.621 ms   node_Conv_293 + node_relu_1
     5.9%    0.620 ms   node_Conv_297 + node_relu_3

TRT int8: 융합 후 레이어 23개, median 합계 5.24 ms
   Reformatting 1개 = 4.0% (순수 오버헤드)
    13.1%    0.684 ms   node_Conv_291 + node_relu + node_max_pool2d
     5.9%    0.309 ms   node_Conv_295 + node_add + node_relu_2
     5.9%    0.308 ms   node_Conv_299 + node_add_1 + node_relu_4
  

## 종합 비교

실측 예시 (이 Jetson, batch=16, GPU Compute Time mean):

| 구성 | 실행 | 속도 (ms/batch16) | 크기 | PyTorch eager 대비 |
|---|---|---|---|---|
| PyTorch FP32 eager | GPU | ~56.0 | 44.6 MB | 1.0x |
| TensorRT FP32 | GPU | ~27.8 | 46.9 MB | **2.0x** |
| TensorRT FP16 | GPU | ~10.6 | 23.6 MB | **5.3x** |
| **TensorRT INT8** | GPU | **~5.4** | **12.0 MB** | **~10.3x** |
| torchvision INT8 | CPU | ~906 | 10.7 MB | (CPU, 느림) |

아래 셀에서 이번 실행의 실제 측정값으로 표를 만듭니다.

In [ ]:
import pandas as pd
rows = [{"구성": "PyTorch FP32 eager", "device": "GPU", "ms/batch16": round(eager_ms,1), "크기(MB)": 44.6}]
for r in trt_results:
    rows.append({"구성": r["name"], "device": "GPU", "ms/batch16": round(r["ms"],1) if r["ms"] else None,
                 "크기(MB)": round(r["size_mb"],1) if r["size_mb"] else None})
rows.append({"구성": "torchvision INT8", "device": "CPU", "ms/batch16": round(int8_ms,1), "크기(MB)": round(sd_size_mb(qm.state_dict()),1)})
df = pd.DataFrame(rows)
base = rows[0]["ms/batch16"]
df["eager대비"] = df["ms/batch16"].apply(lambda v: f"{base/v:.1f}x" if v else "-")
df

## 정리

1. **경량화 ≠ 자동 속도향상**: CPU INT8은 모델 크기를 1/4로 줄이지만 이 Jetson CPU에서는 오히려 느렸다. 이득은 "하드웨어에 최적화된 커널이 있을 때"만 나온다.
2. **엣지 GPU 추론의 정석은 TensorRT**: 같은 ResNet-18을 TensorRT로 변환하니 FP32만으로도 eager 대비 2배, FP16 5배, INT8 ~10배 빨라지고 엔진 크기도 절반~1/4로 줄었다. 다만 이 배율은 `GPU Compute Time`(연산만) 기준이고, 전송까지 포함한 `Latency` 기준으로는 각각 1.96x/4.93x/9.26x로 다소 낮아진다 — 전송 오버헤드는 정밀도와 무관하게 거의 고정값(~0.6~0.7ms)이라, 연산이 빨라질수록(INT8) 전체 시간에서 전송이 차지하는 비중이 커진다(FP32 2.4% → INT8 10.4%).
3. **정밀도 계단**: FP32 → FP16 → INT8로 갈수록 빠르고 작아지지만, FP16/INT8은 정확도 손실 가능성이 있으므로 실제 배포 전 검증 데이터로 정확도를 확인해야 한다. FP16 도약(2.62x)이 이론적 2배를 넘는 건 Orin의 텐서 코어가 FP16부터 본격 동원되기 때문.
4. **이 Jetson 특유의 주의점**: torchvision `--no-deps` 설치, 양자화 백엔드 `qnnpack` 강제, ONNX 외부 데이터 파일 동반, 측정 전 GPU 워밍업(CC8.7 PTX JIT), **ONNX opset 버전 요청이 무시되고 실제로는 18로 저장됨**(B-2 참고).

### 직접 해보기
- `MobileNetV3`/`ResNet50` 등 다른 백본으로 바꿔 크기/속도 트레이드오프 비교
- trtexec에 `--calib`(캘리브레이션 캐시)로 **정확도를 지키는 INT8** 만들어 보기
- `--shapes`로 배치 크기(1, 8, 32)를 바꿔가며 처리량(qps) vs 지연(latency) 관찰
- ImageNet 검증 샘플로 FP16/INT8 엔진의 **top-1 정확도 손실** 실측
- `nsys profile /usr/bin/trtexec --loadEngine=...`로 H2D 전송과 커널 실행이 타임라인에서 얼마나 겹치는지 시각화 — 위 2번에서 짚은 INT8의 전송 비중(~10%) 구간을 눈으로 확인
- (선택) NVIDIA의 **CUDA MCP Server**를 코딩 에이전트에 연결해두면 이후 CUDA/TensorRT 레벨 질문에 최신 문서 기반 답을 받을 수 있음. NVIDIA Developer 계정으로 최초 1회 인증 필요.
  ```
  claude mcp add --scope user --transport http nvidia-cuda-docs https://api.copilot.nsight.ngc.nvidia.com/mcp/cuda-docs
  ```

### 참고 자료
- [PyTorch: Quantized Transfer Learning for CV](https://docs.pytorch.org/tutorials/intermediate/quantized_transfer_learning_tutorial.html)
- [PyTorch: Pruning Tutorial](https://docs.pytorch.org/tutorials/intermediate/pruning_tutorial)
- [PyTorch: Knowledge Distillation Tutorial](https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html)
- [Torchvision Quantized Models](https://docs.pytorch.org/vision/main/models/resnet_quant.html)